In [ ]:
library(Seurat)
library(ggplot2)
library(ggpubr)
library(rcartocolor)
library(pheatmap)
library(dplyr)
library(RColorBrewer)
library(clustree)
getwd()

dataset_id <- "10xMouse_RA"
dir.create("figures_10xMouse_RA")
dir.create("data_10xMouse_RA")

In [ ]:
colorDict = c("Differentiating"="#88535A",
              "2CLC"="#EF8264",
              "Pluripotent"="#F2CC8F")

getwd()


# Pre

In [ ]:

conversionTable <- read.table("annotation/annotation_mm10_conversion_withAge.tsv") # created with annotation_scripts/create_annotations_human.Rmd
head(conversionTable)

In [ ]:
SoloTE_path <- "/mnt/volume_1p5T/results/SoloTEout/10xMouse_RA/LIF/LIF_SoloTE_output/LIF_legacytes_MATRIX"

# get filtered barcodes
STAR_path <- "/mnt/TEresults/snakemake_results/results/STARoutdir/10xMouse_RA/LIF/best_Solo.out/Gene"
filteredBarcodes <- read.table(paste0(STAR_path, "/filtered/barcodes.tsv"))$V1 # read barcodes seleced by STARsolo

# 2CLC sample
legacyTEmatrix <- Seurat::ReadMtx(mtx = paste0(SoloTE_path, "/matrix.mtx"), 
                              cells = paste0(SoloTE_path, "/barcodes.tsv"), 
                              features = paste0(SoloTE_path, "/features.tsv")) # read matrix

legacyTEmatrix <- legacyTEmatrix[,filteredBarcodes]

# select TEs 
TEs <- grep("SoloTE", rownames(legacyTEmatrix), value = T)
locusTEs <- grep("chr", TEs, value = T)
# subset the matrix keeping only TEs
TEmatrix <- legacyTEmatrix[locusTEs,]
# remove "SoloTE" from the name of the TEs

rownames(TEmatrix) <- gsub("SoloTE\\|", "", rownames(TEmatrix))

# rownames(TEmatrix) <- gsub("\\|", "-", rownames(TEmatrix))
# rownames(TEmatrix) <- gsub("\\_", "-", rownames(TEmatrix))
# rownames(TEmatrix) <- gsub("\\?", "", rownames(TEmatrix))

# table(rownames(TEmatrix) %in% conversionTable$soloteID)
# # transform into Stellarscope IDs
# rownames(TEmatrix) <- conversionTable$stellarscopeID[match(rownames(TEmatrix), conversionTable$soloteID)]

nCells <- ncol(TEmatrix)
thrMinCells <- round(nCells * 0.02)
# create Seurat object with shallow filtering of TEs expressed in at least 50 cells and cells expressing at least 50 TEs 
objTE <- Seurat::CreateSeuratObject(TEmatrix, project = "SoloTE", 
            min.cells = thrMinCells, min.features = 50) 
objTE
thrMinCells

In [ ]:
objTE_SoloTE <- objTE#[,filteredBarcodes] # keep only filtered barcodes
objTE_SoloTE

In [ ]:
# select genes
genes <- setdiff(rownames(legacyTEmatrix), TEs)
# subset the matrix keeping only genes
Genematrix <- legacyTEmatrix[genes,]
# create Seurat object with shallow filtering of genes expressed in at least 50 cells and cells expressing at least 100 genes 
Gene <- Seurat::CreateSeuratObject(Genematrix, project = "2CLC", 
                          min.cells = thrMinCells, min.features = 100) 


objGenes_SoloTE <- Gene #[,filteredBarcodes] # keep only filtered barcodes
objGenes_SoloTE

In [ ]:
# QC TEs

options(repr.plot.width=7, repr.plot.height=6)

objTE_SoloTE@meta.data$nCount_TE <- objTE_SoloTE@meta.data$nCount_RNA 
objTE_SoloTE@meta.data$nFeature_TE <- objTE_SoloTE@meta.data$nFeature_RNA 

# Visualize QC metrics as a violin plot
VlnPlot(objTE_SoloTE, features = c("nCount_RNA", "nFeature_RNA"), ncol = 2, 
        cols = colorDict, pt.size = 0, alpha = 0.5) + geom_hline(yintercept = 400)
summary(objTE_SoloTE$nCount_RNA)
summary(objTE_SoloTE$nFeature_RNA)

In [ ]:
# QC genes
options(repr.plot.width=8, repr.plot.height=5)

VlnPlot(objGenes_SoloTE, features = c( "nCount_RNA", "nFeature_RNA" ), ncol = 2, 
        cols = alpha(colorDict, 0.5), pt.size = 0.05, alpha = 0.5) + 
  geom_hline(yintercept = 2000)

summary(objGenes_SoloTE$nFeature_RNA)
summary(objGenes_SoloTE$nCount_RNA)

# Genes

In [ ]:
gc()

objGenes_SoloTE <- NormalizeData(objGenes_SoloTE, normalization.method = "LogNormalize", scale.factor = 10000)

objGenes_SoloTE <- FindVariableFeatures(objGenes_SoloTE, selection.method = "vst", nfeatures = 4000)

# Identify the 10 most highly variable genes
top10 <- head(VariableFeatures(objGenes_SoloTE), 10)

# plot variable features with and without labels
plot1 <- VariableFeaturePlot(objGenes_SoloTE)
plot2 <- LabelPoints(plot = plot1, points = top10, repel = TRUE)
plot2

In [ ]:
gc()
all.genes <- rownames(objGenes_SoloTE)
objGenes_SoloTE <- ScaleData(objGenes_SoloTE) # on hvgs

objGenes_SoloTE <- RunPCA(objGenes_SoloTE, features = VariableFeatures(object = objGenes_SoloTE))


DimPlot(objGenes_SoloTE, reduction = "pca") + NoLegend()

ElbowPlot(objGenes_SoloTE)


In [ ]:
objGenes_SoloTE <- FindNeighbors(objGenes_SoloTE, dims = 1:11, k.param = 20)
objGenes_SoloTE <- FindClusters(objGenes_SoloTE, resolution = 1, algorithm = 4)
objGenes_SoloTE <- RunUMAP(objGenes_SoloTE, dims = 1:11)
DimPlot(objGenes_SoloTE, reduction = "umap")

In [ ]:

options(repr.plot.width=6, repr.plot.height=5)

DimPlot(objGenes_SoloTE, reduction = "umap", group.by="RNA_snn_res.1",
        #cols=colorDict, 
        shuffle=T, pt.size = 0.5) + 
        theme_pubr() +
  theme(text=element_text(size=20)) 
ggsave("figures_10xMouse_RA/umap_genes_clusters.pdf", device = "pdf", width=6, height=5)

In [ ]:
options(repr.plot.width=10, repr.plot.height=14)

FeaturePlot(objGenes_SoloTE, reduction = "umap", features=c("Nanog", "Sox2", "Zfp42",
                                                             "Zscan4a","Zscan4d","Zscan4c",
                                                            "Sox17", "Sox7", "Gata6"),
        pt.size = 0.5) &
        theme_pubclean() &
  theme(text=element_text(size=18)) 
  ggsave("figures_10xMouse_RA/umap_genes_markerExpression.pdf", device = "pdf", width=10, height=14)

In [ ]:
Idents(objGenes_SoloTE) <- as.character(objGenes_SoloTE$`RNA_snn_res.1`)

objGenes_SoloTE <- RenameIdents(objGenes_SoloTE, '1' = 'Pluripotent')
objGenes_SoloTE <- RenameIdents(objGenes_SoloTE, '2' = 'Pluripotent')
objGenes_SoloTE <- RenameIdents(objGenes_SoloTE, '4' = 'Pluripotent')
objGenes_SoloTE <- RenameIdents(objGenes_SoloTE, '5' = 'Pluripotent')


objGenes_SoloTE <- RenameIdents(objGenes_SoloTE, '3' = '2CLC')
objGenes_SoloTE <- RenameIdents(objGenes_SoloTE, '6' = '2CLC')
objGenes_SoloTE <- RenameIdents(objGenes_SoloTE, '7' = '2CLC')


objGenes_SoloTE <- RenameIdents(objGenes_SoloTE, '8' = 'Differentiating')

objGenes_SoloTE$celltype <- Idents(objGenes_SoloTE) 


In [ ]:

options(repr.plot.width=6, repr.plot.height=5)

DimPlot(objGenes_SoloTE, reduction = "umap", group.by="celltype",
        cols=colorDict, 
        shuffle=T, pt.size = 0.5) + 
        theme_pubr() +
  theme(text=element_text(size=20)) 
ggsave("figures_10xMouse_RA/umap_genes_celltypes.pdf", device = "pdf")

In [ ]:
objGenes_SoloTE <- JoinLayers(objGenes_SoloTE)


# TEs 

In [ ]:
objTE_SoloTE@assays$RNA

In [ ]:

### Normalize

objTE_SoloTE <- NormalizeData(objTE_SoloTE, normalization.method = "LogNormalize", scale.factor = 10000)

#saveRDS(objTE_SoloTE, file=paste0("data_",dataset_id,"/objTE_SoloTE_beforeHVG.RDS"))



In [ ]:

objTE_SoloTE <- FindVariableFeatures(objTE_SoloTE, selection.method = "vst", nfeatures = 4000)

# # Identify the 10 most highly variable genes
top10 <- head(VariableFeatures(objTE_SoloTE), 10)

# plot variable features with and without labels
plot1 <- VariableFeaturePlot(objTE_SoloTE)
plot2 <- LabelPoints(plot = plot1, points = top10, repel = TRUE)
plot2

In [ ]:

gc()
all.genes <- rownames(objTE_SoloTE)
objTE_SoloTE <- ScaleData(objTE_SoloTE) # on hvgs

grep("MERVL", all.genes, value = T)[1:50]


In [ ]:

objTE_SoloTE <- RunPCA(objTE_SoloTE, features = VariableFeatures(object = objTE_SoloTE))


DimPlot(objTE_SoloTE, reduction = "pca") + NoLegend()

ElbowPlot(objTE_SoloTE)

In [ ]:
objTE_SoloTE <- FindNeighbors(objTE_SoloTE, dims = 1:12, k.param = 20)
objTE_SoloTE <- FindClusters(objTE_SoloTE, resolution = 1, algorithm=4)
objTE_SoloTE <- RunUMAP(objTE_SoloTE, dims = 1:12)
DimPlot(objTE_SoloTE, reduction = "umap")

In [ ]:
options(repr.plot.width=6, repr.plot.height=5)

DimPlot(objTE_SoloTE, reduction = "umap", group.by = "seurat_clusters",
        shuffle=T, pt.size = 0.5) + 
  theme_pubr() +
  scale_color_carto_d(palette = "Pastel") +
  theme(text=element_text(size=20)) 
ggsave("figures_10xMouse_RA/umap_locus_clusters.pdf", device = "pdf", width=6, height=5)


In [ ]:
options(repr.plot.width=6, repr.plot.height=5)

# transfer celltype annotation from genes
objTE_SoloTE$celltype <- objGenes_SoloTE$celltype[Cells(objTE_SoloTE)]

options(repr.plot.width=6, repr.plot.height=5)

DimPlot(objTE_SoloTE, reduction = "umap", group.by = "celltype",
        cols=colorDict, 
        shuffle=T, pt.size = 0.5) + 
  theme_pubr() +
  theme(text=element_text(size=20), plot.title=element_text(hjust=0.5)) 
ggsave(paste0("figures_",dataset_id,"/umap_locus_SoloTE_celltypes.pdf"), device = "pdf", width=6, height=5)


In [ ]:
options(repr.plot.width=12, repr.plot.height=5)

FeaturePlot(objTE_SoloTE, features = c('chr10-105000001-105001301-MERVL-int:ERVL:LTR-1.1--',
  'chr10-107130124-107135454-MERVL-int:ERVL:LTR-1.5-+'))

In [ ]:
# Comparison to genes

options(repr.plot.width=8, repr.plot.height=8)


objGenes_SoloTE$celltype_num <- objGenes_SoloTE$seurat_clusters
levels(objGenes_SoloTE$celltype_num) <- 1:length(levels(objGenes_SoloTE$seurat_clusters))                          

# Add cluster information from objTE_clustered to objGenes_clustered
objGenes_SoloTE$clusters.0.1 <- objGenes_SoloTE$seurat_clusters
objGenes_SoloTE$clusters.0.2 <- objTE_SoloTE$seurat_clusters
objGenes_SoloTE$clusters.0.3 <- objGenes_SoloTE$celltype

concordanceTable <- table(objGenes_SoloTE$seurat_clusters, objTE_SoloTE$seurat_clusters)

pheatmap(concordanceTable, display_numbers = T, color = colorRampPalette(brewer.pal(9,'Blues')[1:6])(100),
         border_color = NA, number_format = "%.0f", cellwidth = 40, cellheight = 40, fontsize = 25)

# Generate the clustree plot
clustree(objGenes_SoloTE, prefix = "clusters.", node_text_angle=0, node_text_size=5) + theme(text=element_text(size=15)) + 
  scale_color_manual(values=alpha(c("#81B29A","#8A8BA8","#F2CC8F"), 0.6)) + scale_size(range = c(3,20)) +
  guides(colour = FALSE) 
ggsave("figures_10xMouse_RA/clustree_clusters_allcelltypes.pdf")

objTE_SoloTE$RNAclusters <- objGenes_SoloTE$seurat_clusters


In [ ]:
options(repr.plot.width=6, repr.plot.height=5)

DimPlot(objTE_SoloTE, reduction = "umap", group.by = "RNAclusters",
        shuffle=T, pt.size = 0.5) + 
  theme_pubr() +
  theme(text=element_text(size=20)) 
ggsave("figures_10xMourse_RA/umap_locus_RNAclusters_SoloTE.pdf", device = "pdf")


In [ ]:
clustersResList <- list()
identical(Cells(objGenes_SoloTE), Cells(objTE_SoloTE)) # TRUE, same cells in same order

for(res in seq(0.5, 2, by=0.1)){
    print(res)
    
    clusters_genes <- FindClusters(objGenes_SoloTE, resolution = res)
    clusters_TEs <- FindClusters(objTE_SoloTE, resolution = res)

    clustersResList[[as.character(res)]] <- cbind(clusters_genes$seurat_clusters, clusters_TEs$seurat_clusters)
    #
}


In [ ]:
library(mclust)
library(viridis)

resolutions <- seq(0.5, 2.0, by = 0.1)

ari_matrix <- matrix(data=NA, nrow=length(resolutions), ncol=length(resolutions))
colnames(ari_matrix) <- paste0("gene_r",resolutions)
rownames(ari_matrix) <- paste0("TE_r",resolutions)

for(r_gene in resolutions){
    for(r_TE in resolutions){
        ari <- adjustedRandIndex(clustersResList[[as.character(r_gene)]][,1],
                    clustersResList[[as.character(r_TE)]][,2])
        ari_matrix[paste0("TE_r",r_TE),paste0("gene_r",r_gene)] <- ari
    }
}
ari_matrix

p <- pheatmap(ari_matrix, display_numbers = T, color = mako(100, alpha = 1, begin = 0, end = 1, direction = 1)[],
         border_color = NA, number_format = "%.3f", number_color="black",
         cluster_rows = FALSE, cluster_columns = FALSE,
         cellwidth = 25, cellheight = 25)

pdf(paste0("figures_",dataset_id,"/ARI_celltypes_SoloTE.pdf"), width=9.5, height=9)
p
dev.off()
# ari_values <- sapply(resolutions, function(r) {
#     adjustedRandIndex(clustersResList[[as.character(r)]][,1],
#                       clustersResList[[as.character(r)]][,2])
# })

# plot(resolutions, ari_values, type = "b",
#      xlab = "Resolution", ylab = "ARI",
#      main = "Gene vs Transposon Clustering Similarity")


In [ ]:
res <- "0.8"
TEclusters <- clustersResList[[res]][,2]
table(clustersResList[[res]][,2])
objTE_SoloTE <- FindClusters(objTE_SoloTE, resolution = as.numeric(res))

res <- "1"
GENEclusters <- clustersResList[[res]][,1]
table(clustersResList[[res]][,1])
objGenes_SoloTE <- FindClusters(objGenes_SoloTE, resolution = as.numeric(res))


objTE_SoloTE$RNAclusters <- objGenes_SoloTE$seurat_clusters
objGenes_SoloTE$TEclusters <- objTE_SoloTE$seurat_clusters

concordanceTable <- table(objGenes_SoloTE$seurat_clusters, objTE_SoloTE$seurat_clusters)
rownames(concordanceTable) <- paste("GeneCluster_",rownames(concordanceTable))
colnames(concordanceTable) <- paste("TECluster_",colnames(concordanceTable))

In [ ]:
options(repr.plot.width=6, repr.plot.height=5)

DimPlot(objTE_SoloTE, reduction = "umap", group.by = "seurat_clusters",
        shuffle=T, pt.size = 0.5) + 
  theme_pubr() +
  scale_color_carto_d(palette = "Pastel") +
  theme(text=element_text(size=20)) 
ggsave("figures_10xMouse_RA/umap_locusTE_SoloTE_seurat_clusters.pdf", device = "pdf")

DimPlot(objTE_SoloTE, reduction = "umap", group.by = "RNAclusters",
        shuffle=T, pt.size = 0.5) + 
  theme_pubr() +
  scale_color_brewer(palette="Spectral") +
  theme(text=element_text(size=20)) 
ggsave("figures_10xMouse_RA/umap_locusTE_SoloTE_RNAclusters.pdf", device = "pdf")

DimPlot(objGenes_SoloTE, reduction = "umap", group.by = "TEclusters",
        shuffle=T, pt.size = 0.5) + 
  theme_pubr() +
  scale_color_carto_d(palette = "Pastel") +
  theme(text=element_text(size=20)) 
ggsave("figures_10xMouse_RA/umap_genes_SoloTE_TEclusters.pdf", device = "pdf")

DimPlot(objGenes_SoloTE, reduction = "umap", group.by = "seurat_clusters",
        shuffle=T, pt.size = 0.5) + 
  theme_pubr() +
  scale_color_brewer(palette="Spectral") +
  theme(text=element_text(size=20)) 
ggsave("figures_10xMouse_RA/umap_genes_SoloTE_RNAclusters.pdf", device = "pdf")

In [ ]:
library(clue)
library(grid)
options(repr.plot.width=9, repr.plot.height=8)

# Suppose your matrix has more rows than columns
nr <- nrow(concordanceTable)
nc <- ncol(concordanceTable)

if(nr > nc){
  # pad with zeros to make it square
  M <- cbind(concordanceTable, matrix(0, nrow = nr, ncol = nr - nc))
} else {
  M <- rbind(concordanceTable, matrix(0, nrow = nc - nr, ncol = nc))
}

perm <- solve_LSAP(M, maximum = TRUE)

# keep only the original columns
M_reordered <- M[, perm[1:nc]]
rownames(M_reordered)


ph <- pheatmap( M_reordered[(rownames(M_reordered)!=""),!is.na(colnames(M_reordered))] , 
        display_numbers = T, color = colorRampPalette(brewer.pal(9,'Blues')[1:7])(100),#adjustcolor((colorRampPalette(carto_pal(name="Emrld")))(100),alpha.f=1),
        main = "Cluster concordance - TEs VS Genes",
        cluster_rows=FALSE, cluster_cols=FALSE, 
                fontsize_number = 14, fontsize = 14, annotation_names_row = TRUE, number_color = "black",
         border_color = NA, number_format = "%.0f", cellwidth = 30, cellheight = 30)


# write to PDF safely

pdf(paste0("figures_", dataset_id, "/heatmap_cluster_clusters_concordance_SoloTE.pdf"), width=8, height=9)

grid::grid.newpage()
grid::grid.draw(ph$gtable)

dev.off()


In [ ]:
# # Save objects
# saveRDS(objGenes_SoloTE, paste0("data_", dataset_id, "/soloTE_", dataset_id, "_GENE_seuratObj_251124.RDS"))
# saveRDS(objTE_SoloTE, paste0("data_", dataset_id, "/soloTE_", dataset_id, "_seuratObj_251124.RDS"))

# Checkpoint

In [ ]:
#save.image("workspaces/10xMouse_RA_SoloTE_251124.Rdata")
load("workspaces/10xMouse_RA_SoloTE_251124.Rdata")

In [ ]:
options(repr.plot.width=8, repr.plot.height=4)
library(reshape2)
concordanceTable <- table(objTE_SoloTE$seurat_clusters, objTE_SoloTE$celltype)

df <- melt(concordanceTable)
df <- df[df$value!=0,]
colnames(df) <- c("cluster","celltype","nCells")
df$cluster <- as.character(df$cluster)
df$clusterSize <- table(objTE_SoloTE$seurat_clusters)[df$cluster]
df$percentage <- as.numeric(df$nCells / df$clusterSize *100)

# df <- df %>%
#   group_by(cluster) %>%
#   mutate(main = celltype[which.max(percentage)]) %>%
#   ungroup() %>%
#   arrange(main, desc(percentage))
# df$cluster <- factor(df$cluster, levels = unique(df$cluster))


ggplot(df, aes(x=cluster, y=nCells, fill=celltype)) + 
  geom_col() + 
  xlab("TE cluster") +
  scale_fill_manual(values=colorDict)+
  theme_minimal() + theme(text=element_text(size=20))
ggsave(paste0("figures_", dataset_id, "/clusterIdentity_barplot_SoloTE_noplatelet.pdf"), width=8.5, height=4)